# R1-06 · Estadística descriptiva — Profundización (opcional) 🔬

**Formación Pública — Rama R1 · Analista de Datos · Notebook de profundización**

Opcional y de **transferencia**: las mismas herramientas sobre otra variable — la **población de
30 comunas** de muy distintos tamaños (INE, Censo 2024). Medidas nuevas: **robustez** (qué pasa
al sacar un extremo) y **puntaje z**, comparado con la regla del IQR.

> Ojo: son 30 comunas elegidas para cubrir distintos tamaños, **no** todas las comunas del país:
> los resúmenes describen esta muestra, no «la comuna chilena».

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, os, urllib.request
CSV = "comunas.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/comunas.csv", CSV)
com = pd.read_csv(CSV)   # 30 comunas de muy distintos tamaños (INE, Censo 2024)
print(com.shape)
com["poblacion"].describe()

## 1. Centro y sesgo en otra variable

Dominio nuevo: la **población** de 30 comunas de muy distintos tamaños (Censo 2024), desde Puente Alto hasta Pichilemu. ¿Se repite el patrón de la matrícula?

### ✍️ Ejercicio 1 — Razón media/mediana de la población

In [ ]:
razon = com["poblacion"].mean() / com["poblacion"].median()
asimetrica_derecha = razon > 1.15
print(f"media: {com['poblacion'].mean():,.0f} | mediana: {com['poblacion'].median():,.0f} | razón: {razon:.2f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert abs(razon - com["poblacion"].mean() / com["poblacion"].median()) < 1e-9
    assert asimetrica_derecha == (razon > 1.15)
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'com["poblacion"].mean() / com["poblacion"].median() y la comparación razon > 1.15.')
    print("   Detalle:", e)

## 2. Robustez: ¿qué medida se mueve si saco una comuna?

Una medida **robusta** casi no cambia si quitas un valor extremo. Saca la comuna más poblada y mide cuánto cambian (en %) la media y la mediana.

### ✍️ Ejercicio 2 — Quita la comuna más grande y compara

In [ ]:
sin_mayor = com[com["poblacion"] != com["poblacion"].max()]
cambio_media = 100 * (sin_mayor["poblacion"].mean() - com["poblacion"].mean()) / com["poblacion"].mean()
cambio_mediana = 100 * (sin_mayor["poblacion"].median() - com["poblacion"].median()) / com["poblacion"].median()
print(f"la media cambia {cambio_media:.1f}% | la mediana cambia {cambio_mediana:.1f}%")

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    _m0, _m1 = com["poblacion"].mean(), sin_mayor["poblacion"].mean()
    _d0, _d1 = com["poblacion"].median(), sin_mayor["poblacion"].median()
    assert abs(cambio_media - 100 * (_m1 - _m0) / _m0) < 1e-9
    assert abs(cambio_mediana - 100 * (_d1 - _d0) / _d0) < 1e-9
    assert abs(cambio_media) > abs(cambio_mediana), "La media debería moverse más que la mediana"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '100 * (nuevo - original) / original, para .mean() y para .median().')
    print("   Detalle:", e)

## 3. Atípicos con la regla del IQR

Aplica la misma regla de la lección: atípico = sobre `Q3 + 1,5 × IQR`. ¿Qué comunas lo son?

### ✍️ Ejercicio 3 — Comunas atípicas

In [ ]:
q1, q3 = com["poblacion"].quantile([0.25, 0.75])
limite = q3 + 1.5 * (q3 - q1)
atipicas = sorted(com.loc[com["poblacion"] > limite, "comuna"])
print(f"límite: {limite:,.0f} habitantes | atípicas: {atipicas}")

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    _q1, _q3 = com["poblacion"].quantile([0.25, 0.75])
    _lim = _q3 + 1.5 * (_q3 - _q1)
    assert abs(limite - _lim) < 1e-6
    assert list(atipicas) == sorted(com.loc[com["poblacion"] > _lim, "comuna"])
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'limite = q3 + 1.5 * (q3 - q1); sorted(com.loc[com["poblacion"] > limite, "comuna"]).')
    print("   Detalle:", e)

## 4. Puntaje z: ¿a cuántas desviaciones del promedio?

El **puntaje z** = (valor − media) / desviación dice cuán lejos está cada dato del promedio, en «desviaciones». Un |z| > 2 suele considerarse lejano. Pero cuidado: con datos asimétricos la media y la desviación ya están infladas por los extremos, así que el z **detecta menos** atípicos que la regla IQR.

### ✍️ Ejercicio 4 — Calcula z y cuenta los lejanos

In [ ]:
com["z"] = (com["poblacion"] - com["poblacion"].mean()) / com["poblacion"].std()
n_lejanos = int((com["z"].abs() > 2).sum())
comuna_max_z = com.loc[com["z"].idxmax(), "comuna"]
print(com.sort_values("z", ascending=False).head(4)[["comuna", "poblacion", "z"]].round(2))
print("comunas con |z| > 2:", n_lejanos, "| mayor z:", comuna_max_z)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    _z = (com["poblacion"] - com["poblacion"].mean()) / com["poblacion"].std()
    assert abs(com["z"] - _z).max() < 1e-9, "z = (poblacion - media) / desviación"
    assert n_lejanos == int((_z.abs() > 2).sum())
    assert comuna_max_z == com.loc[_z.idxmax(), "comuna"]
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '(com["poblacion"] - com["poblacion"].mean()) / com["poblacion"].std(); luego .abs() > 2 y idxmax().')
    print("   Detalle:", e)

---

*Fin de la profundización de R1-06.*